# Re-evaluate Existing Model

Score a logged model against a labeled eval dataset and upsert the result on the trial run. This uses the same `evaluate(...)` path as trial-time evaluation: per-label reports and predictions are written through the trial artifact seam.


In [ ]:
from __future__ import annotations

import os

import numpy as np
import pandas as pd
from IPython.display import display

import automl
from automl import data, eval, experiment, trial
from automl.eval import Auc, EvalSpec, LogLoss, Metric, ThresholdSweep

DRY_RUN = True
NAMESPACE = os.getenv("AUTOML_NOTEBOOK_NAMESPACE", "")


In [ ]:
active = automl.use_project(dry_run=DRY_RUN, namespace=NAMESPACE)
config = active.config
display(
    {
        "project": active.project_name,
        "repo_root": str(config.repo_root),
        "project_dir": str(config.project_dir),
        "experiment": active.active_experiment_id,
        "dry_run": active.dry_run,
        "namespace": active.namespace or "<none>",
    }
)

leaderboard = experiment.leaderboard(training_origin="all", n=20, session=active)
rows = list(leaderboard.rows)
if not rows:
    raise RuntimeError("run notebook 3.1 with RUN_AGENT=True or notebook 3.2 with RUN_TRIAL=True first")

deployed = rows[0]
deployed.to_dict()


In [ ]:
loaded = data.materialize(session=active)
run_config = active.config.require_run_config()
train = data.load_dataset(split_name=run_config.train_split, session=active)
holdout = data.load_dataset(split_name=run_config.eval_split, session=active)
target_col = loaded.dataset.target_column
unique_key = tuple(loaded.dataset.unique_key)

{
    "dataset_id": loaded.dataset.id,
    "target_column": target_col,
    "unique_key": unique_key,
}


In [ ]:
eval_dataset, cached = eval.prepare_eval_dataset(
    session=active,
    dataset_id=loaded.dataset.id,
    split=active.config.require_run_config().eval_split,
)

result = eval.evaluate(
    session=active,
    model_run_id=deployed.run_id,
    eval_dataset_id=eval_dataset.id,
    label="notebook_eval",
    overwrite=False,
)

{
    "label": result.label,
    "eval_dataset_id": result.eval_dataset_id,
    "metrics": result.metrics,
    "cached_dataset": cached,
    "cached_result": result.cached,
    "predictions_uri": result.predictions_uri,
}


In [ ]:
metrics_result = eval.evaluate(
    session=active,
    model_run_id=deployed.run_id,
    eval_dataset_id=eval_dataset.id,
    eval_spec=EvalSpec(
        primary=Auc(),
        metrics=[
            -LogLoss(),
            ThresholdSweep(thresholds=[0.3, 0.5, 0.7]),
        ],
    ),
    label="notebook_eval_custom_metrics",
    overwrite=False,
)

{
    "label": metrics_result.label,
    "primary": metrics_result.primary,
    "scalar_metrics": metrics_result.metrics,
    "cached_result": metrics_result.cached,
}


In [ ]:
primary_result = eval.evaluate(
    session=active,
    model_run_id=deployed.run_id,
    eval_dataset_id=eval_dataset.id,
    eval_spec=EvalSpec(
        primary=-LogLoss(),
        metrics=[
            Auc(),
            ThresholdSweep(thresholds=[0.3, 0.5, 0.7]),
        ],
    ),
    label="notebook_eval_primary_logloss",
    set_as_primary_label=True,
    overwrite=False,
)

{
    "label": primary_result.label,
    "primary": primary_result.primary,
    "primary_label": primary_result.label,
    "cached_pointer_update": primary_result.cached,
}


In [ ]:
class WeightedMeanScore(Metric):
    required_augmentations = ("risk_weight",)
    required_columns = ("RISK_WEIGHT",)

    def compute(self, df_test, y_pred, target_col):
        return float(np.average(y_pred, weights=df_test["RISK_WEIGHT"]))


risk_weight_frame = holdout.df.loc[:, list(unique_key)].copy()
risk_weight_frame["RISK_WEIGHT"] = np.linspace(1.0, 2.0, len(risk_weight_frame))
risk_weight, risk_weight_cached = eval.prepare_eval_augmentation(
    session=active,
    eval_dataset_id=eval_dataset.id,
    frame=risk_weight_frame,
    name="risk_weight",
)

augmented_result = eval.evaluate(
    session=active,
    model_run_id=deployed.run_id,
    eval_dataset_id=eval_dataset.id,
    eval_spec=EvalSpec(primary=WeightedMeanScore(), metrics=[Auc()]),
    label="notebook_eval_with_risk_weight",
    overwrite=False,
)

{
    "augmentation": risk_weight.data_gcs_uri,
    "augmentation_cached": risk_weight_cached,
    "label": augmented_result.label,
    "primary": augmented_result.primary,
    "metrics": augmented_result.metrics,
    "cached_result": augmented_result.cached,
}


In [ ]:
external_parts = [
    group.head(min(10, len(group)))
    for _, group in train.df.groupby(target_col, sort=True)
]
external_frame = pd.concat(external_parts, ignore_index=True)
if external_frame[target_col].nunique() < 2:
    raise RuntimeError("external eval sample must contain both target classes for AUC")

external_eval, external_cached = eval.prepare_eval_dataset(
    session=active,
    kind="external",
    frame=external_frame,
    target_col=target_col,
    unique_key=unique_key,
    provenance={"source": "notebook_5_external_labeled_sample"},
)

external_result = eval.evaluate(
    session=active,
    model_run_id=deployed.run_id,
    eval_dataset_id=external_eval.id,
    eval_spec=EvalSpec(primary=Auc(), metrics=[-LogLoss()]),
    label="notebook_external_labeled_sample",
    overwrite=False,
)

{
    "eval_dataset_id": external_result.eval_dataset_id,
    "kind": external_eval.kind,
    "cached_dataset": external_cached,
    "label": external_result.label,
    "metrics": external_result.metrics,
    "predictions_uri": external_result.predictions_uri,
}


In [ ]:
details = trial.show_trial(deployed.run_id, session=active)
details.evaluations
